In [ ]:
!which pip

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
from collections import Counter
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import fasttext

In [ ]:
builder = (SparkSession.builder 
    .appName("MyDeltaApp") 
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") 
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.LocalLogStore") 
    .config("spark.driver.memory", "8g")               # Memory for the Driver
    .config("spark.executor.memory", "8g")             # Memory per Executor
    .config("spark.executor.cores", "4")               # Number of cores per Executor
    .config("spark.executor.instances", "4")           # Number of Executors (4 Executors x 4 cores = 16 cores)
    .config("spark.driver.maxResultSize", "4g") 
    .config("spark.sql.adaptive.enabled", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark Version:", spark.version)

In [ ]:
df = spark.read.format("delta").load("file:///projectnb/ds340/projects/nathan_song_ze_chang_Final_Project/data/raw/fct_issue_event")

In [5]:
df = df.where("issue_closed_at is not null") # only these are useful

# drop useless cols
df = df.drop("event_id","action","author_association","issue_state","issue_state_reason","issue_comment_count",\
            "issue_updated_at","event_created_at","dim_actor")

df = df.withColumn("issue_resolve_time_days", 
                   F.expr("((cast(issue_closed_at as long) - cast(issue_created_at as long)) / 86400)"))

df = df.where("issue_resolve_time_days > 0.0")

In [6]:
df = df.withColumn(
    "issue_resolve_time_label",
    F.expr("""
        CASE 
            WHEN issue_resolve_time_days < (1 / 24) THEN '<1 hour'
            WHEN issue_resolve_time_days >= (1 / 24) AND issue_resolve_time_days < (6 / 24) THEN '1-6 hours'
            WHEN issue_resolve_time_days >= (6 / 24) AND issue_resolve_time_days < 1 THEN '6-24 hours'
            WHEN issue_resolve_time_days >= 1 AND issue_resolve_time_days < 7 THEN '1-7 days'
            WHEN issue_resolve_time_days >= 7 AND issue_resolve_time_days < 28 THEN '1-4 weeks'
            WHEN issue_resolve_time_days >= 28 AND issue_resolve_time_days < 180 THEN '1-6 months'
            WHEN issue_resolve_time_days >= 180 AND issue_resolve_time_days < 365 THEN '6 months - 1 year'
            WHEN issue_resolve_time_days >= 365 THEN '>1 year'
            ELSE 'Unknown'
        END
    """)
)

In [7]:
# Display a wide number of rows and without truncating columns
df.show(10, truncate=False,vertical=True)

-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
import re
import fasttext
import pandas as pd

import re

def clean_text(text: str) -> str:
    """
    Clean the text by removing unnecessary characters and clutter while preserving content.
    
    Args:
        text (str): The raw text to be cleaned.
        
    Returns:
        str: The cleaned text.
    """
    if text is None:
        return ""
    
    # Convert to lowercase (optional)
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Replace multiple newlines or spaces with a single space
    text = re.sub(r'[\r\n]+', ' ', text)  # Replace newlines with space
    text = re.sub(r'\s+', ' ', text)  # Remove multiple spaces
    
    # Remove special characters (keeping alphanumeric, punctuation, and common symbols)
    text = re.sub(r"[^a-zA-Z0-9.,!?;:@#\-_'\" ]+", '', text)
    
    # Replace multiple dots (e.g., "...") with a single dot.
    text = re.sub(r'\.{2,}', '.', text)
    
    return text.strip()


In [9]:
# Randomly sample 10% of the rows; seed for reproducibility
df_sample = df.sample(fraction=0.1, seed=42)
print("Sampled 10% of data.")

# Convert the 'issue_body' column of the sampled DataFrame to Pandas
pdf = df_sample.select("issue_body").toPandas()
print("Converted sampled DataFrame to Pandas DataFrame.")

# Import FastText and load the model (ensure the model path is correct)
model = fasttext.load_model("./models/lid.176.bin")

Sampled 10% of data.


Converted sampled DataFrame to Pandas DataFrame.


In [12]:
# Apply language detection to each row of 'issue_body'
pdf["language"] = pdf["issue_body"].apply(detect_language)
print("Language detection complete.")

# Count the languages and display the results
language_counts = pdf["language"].value_counts()
top10_languages = language_counts.head(10)
print("Top 10 languages:")
print(top10_languages)


Language detection complete.
Top 10 languages:
language
en         163315
unknown     44542
de          26404
als         20383
it          19261
hu          13228
zh           8081
pt           7854
es           6192
fr           6075
Name: count, dtype: int64


In [ ]:
df.select("issue_body").show(20, truncate=False)


In [ ]:
# Perform the groupBy and count operation
result = df.groupBy("issue_resolve_time_label") \
           .agg(F.count("*").alias("count")) \
           .orderBy(F.desc("count"))

# Show the result (if you want to see it in PySpark terminal)
result.show()

In [ ]:
resolve_time_pd = df.select("issue_resolve_time_label") \
                    .toPandas()

In [ ]:
resolve_time_pd.describe()

In [ ]:
# Count occurrences of each label
label_counts = Counter(resolve_time_pd['issue_resolve_time_label'])

# Sort labels by the intended order of importance
ordered_labels = [
    '<1 hour', '1-6 hours', '6-24 hours', '1-7 days', 
    '1-4 weeks', '1-6 months', '6 months - 1 year', '>1 year'
]

# Prepare data for plotting
counts = [label_counts.get(label, 0) for label in ordered_labels]

# Plot the distribution
plt.figure(figsize=(10, 6))
plt.bar(ordered_labels, counts, color='skyblue')
plt.title('Distribution of Issue Resolution Time Labels')
plt.xlabel('Issue Resolution Time Label')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.show()

In [ ]:
df.show()